In [11]:
import os
import shutil
from tqdm import tqdm

In [12]:
filepath = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Misclassified_data.txt"
with open(filepath, 'r') as f:
    misclassified = [line[:-1] for line in f]
print(len(misclassified))

1504


In [19]:
def move_files(source_dir, dest_dir, file_list):
    for date in os.listdir(source_dir):
        datefolder = os.path.join(source_dir, date)
        for species in tqdm(os.listdir(datefolder), desc=f'{date}'):
            speciesfolder = os.path.join(datefolder, species)
            for type in os.listdir(speciesfolder):                
                typefolder = os.path.join(speciesfolder, type)
                if not os.path.isdir(typefolder): continue
                for imgfolder in os.listdir(typefolder):
                    imgfolderpath = os.path.join(typefolder, imgfolder)
                    if not os.path.isdir(imgfolderpath): continue
                    for img in os.listdir(imgfolderpath):
                        if img in file_list:
                            filepath = os.path.join(imgfolderpath, img)
                            if imgfolder=='Single' or imgfolder=='Group':
                                if not os.path.exists(os.path.join(dest_dir, img)):
                                    shutil.move(filepath, dest_dir)
                                else:
                                    if not os.path.exists(os.path.join(typefolder, 'Misclassified', img)):
                                        shutil.move(filepath, os.path.join(typefolder, 'Misclassified'))
                                    else:
                                        os.remove(filepath)
                                        
                            elif imgfolder=='Misclassified':
                                if not os.path.exists(os.path.join(dest_dir, img)):
                                    shutil.move(filepath, dest_dir)
                
                
# source_dir = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\S3 Daily Data"
source_dir = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Manual Data"
dest_dir = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Misclassified"
move_files(source_dir, dest_dir, misclassified)

2024-02-27: 100%|██████████| 2/2 [00:00<00:00, 27.35it/s]


In [20]:
def replace_misspelled_folder_names(species_name):
    misspelled_folders = {
        'Are' : ['Ar', 'Are'],
        'Basa' : ['Basa', 'Basaa'],
        'Barracuda' : ['Barcoda', 'Barkoda', 'Barracoda', 'Barracuda'],
        'Bolo' : ['Bolo', 'Bulo'],
        'Catla' : ['Katala', 'Katalaa', 'Katla'],
        'Croaker' : ['Kokor', 'Croaker', 'Silver croaker'],
        'Chara pona' : ['Chara pana'],
        'Demo' : ['Demo', 'Demo2', 'Test', 'Trial'],
        'Emperor' : ['Comprel', 'Emperor', 'Emporwel', 'Empowel',
                        'M perl', 'M preal'],
        'Hilsa' : ['Hilsa', 'Hilis', 'Hilisa'],
        'Lady' : ['Lady', 'Ledi'],
        'Malabar trevally' : ['Mabar tavili', 'Malbhot', 'Travely',
                                'Trvili', 'Trevally', 'Travelly'],
        'Needle' : ['Needale', 'Nidal', 'Nidil'],
        'Parsi' : ['Parci'],
        'Pearl spot' : ['(bloch,', 'Bloch,', 'Bloch',
                        'Pearl spot', 'Pearls spot', 'Hols spot',
                        'Green chromide', 'Green chormide',
                        'Hals spot'],
        'Sea bass' : ['C boss', 'C boos', 'Siba'],
        'Shol' : ['Sholo'],
        'Snapper' : ['Sinper', 'Sniper'],
        'White snapar' : ['White snapper'],
    }

    for key, misspellings in misspelled_folders.items():
        if species_name in misspellings:
            return key
    return species_name


def manual_data_to_final_data(source_dir, destination_dir):
    os.makedirs(destination_dir, exist_ok=True)
    for date in os.listdir(source_dir):
        
        date_path = os.path.join(source_dir, date)
        if not os.path.isdir(date_path): continue

        for species in tqdm(os.listdir(date_path),
                            desc=f'Loading {date} data'):
            species = species.capitalize()
            if species.endswith(" "):
                species = species[:-1]
            
            old_species = species
            species = replace_misspelled_folder_names(species)
            
            if old_species != species:
                duplicate_folder_final = os.path.join(destination_dir, old_species)
                duplicate_folder_manual = os.path.join(source_dir, date, old_species)
                if os.path.exists(duplicate_folder_final):
                    shutil.rmtree(duplicate_folder_final)
                if os.path.exists(duplicate_folder_manual):
                    shutil.rmtree(duplicate_folder_manual)
                    
            src_species_path = os.path.join(date_path, species)
            des_species_path = os.path.join(destination_dir, species)
            if not os.path.isdir(src_species_path): continue

            for fresh_type in os.listdir(src_species_path):
                fresh_type = fresh_type.capitalize()
                if fresh_type.endswith(" "):
                    fresh_type = fresh_type[:-1]

                src_fresh_type_path = os.path.join(src_species_path, fresh_type)
                des_fresh_type_path = os.path.join(des_species_path, fresh_type)

                if not os.path.isdir(src_fresh_type_path): continue
                os.makedirs(des_fresh_type_path, exist_ok=True)

                for folder in os.listdir(src_fresh_type_path):
                    folder_path = os.path.join(src_fresh_type_path, folder) 
                                              
                    if folder == 'Misclassified':
                        des_misclassified_path = os.path.join(des_species_path, 'Misclassified')
                        os.makedirs(des_misclassified_path, exist_ok=True)
                        for img in os.listdir(folder_path):
                            img_path = os.path.join(folder_path, img)
                            des_filepath = os.path.join(des_misclassified_path, img)
                            
                            grp_folder_path = os.path.join(des_fresh_type_path, 'Group')
                            single_folder_path = os.path.join(des_fresh_type_path, 'Single')
                            grp_img_path = os.path.join(grp_folder_path, img)
                            single_img_path = os.path.join(single_folder_path, img)
                            if os.path.exists(grp_img_path):
                                os.remove(grp_img_path)
                            elif os.path.exists(single_img_path):
                                os.remove(single_img_path)
                            elif not os.path.exists(des_filepath):
                                shutil.copy(img_path, des_filepath)

                    elif folder == 'Group' or folder == 'Single':
                        des_folder_path = os.path.join(des_fresh_type_path, folder)
                        os.makedirs(des_folder_path, exist_ok=True)
                        for img in os.listdir(folder_path):
                            img_path = os.path.join(folder_path, img)
                            des_filepath = os.path.join(des_folder_path, img)
                            if not os.path.exists(des_filepath):
                                shutil.copy(img_path, des_filepath)
                            existing_img_path = os.path.join(des_fresh_type_path, img)
                            if os.path.exists(existing_img_path):
                                os.remove(existing_img_path)     

source_directory = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Manual Data"
destination_directory = r"C:\Users\sowmy\Downloads\QzenseLabs\qZense Dataset\Final Data"

manual_data_to_final_data(source_directory, destination_directory)

Loading 2024-02-27 data: 100%|██████████| 2/2 [00:00<00:00, 24.82it/s]
